In [ ]:
# 1. Instalar dependencias
!pip -q install -U fastapi uvicorn pyngrok requests langchain langchain-openrouter

In [ ]:
# 2. Importar librerías y configurar secretos
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google.colab import userdata
from langchain_openrouter import ChatOpenRouter

In [ ]:
# Configurar OpenRouter
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise ValueError("Agrega OPENROUTER_API_KEY en los Secrets de Colab.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Crear el modelo (usamos el mismo que en el Capítulo 1)
llm = ChatOpenRouter(
    model="google/gemini-2.5-flash-lite",
    temperature=0.3,
)

In [ ]:
# 3. Crear la aplicación FastAPI
app = FastAPI(
    title="Chatbot con memoria",
    description="API con dos endpoints: /chat y /reset. Mantiene historial por sesión.",
    version="1.0.0",
)

In [ ]:
# (Los modelos Pydantic y los endpoints los agregaremos en los siguientes pasos)
print("✅ Entorno listo. Ahora definiremos los modelos y endpoints.")

In [ ]:
# 4. Modelos Pydantic para el chatbot

class ChatRequest(BaseModel):
    session_id: str
    message: str

class ChatResponse(BaseModel):
    session_id: str
    response: str

class ResetRequest(BaseModel):
    session_id: str

class ResetResponse(BaseModel):
    session_id: str
    status: str

Ejecuta esta celda para crear el almacenamiento:

In [ ]:
# 5. Almacenamiento en memoria
# Diccionario global: clave = session_id, valor = lista de mensajes
memory_store = {}

def get_session_history(session_id: str):
    """Obtiene el historial de una sesión. Si no existe, lo crea."""
    if session_id not in memory_store:
        memory_store[session_id] = []
    return memory_store[session_id]

In [ ]:
# 6. Endpoint POST /chat
@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    try:
        # 1. Obtener el historial de la sesión
        history = get_session_history(request.session_id)

        # 2. Construir la lista de mensajes para LangChain
        messages = [
            {
                "role": "system",
                "content": "Eres un asistente útil, amable y conciso. Responde siempre en español."
            }
        ]

        # Agregar los mensajes anteriores de la conversación
        messages.extend(history)

        # Agregar el nuevo mensaje del usuario
        messages.append({
            "role": "user",
            "content": request.message
        })

        # 3. Llamar al modelo
        response = llm.invoke(messages)

        # 4. Guardar el nuevo intercambio en el historial
        history.append({
            "role": "user",
            "content": request.message
        })
        history.append({
            "role": "assistant",
            "content": response.content
        })

        # 5. Devolver la respuesta
        return ChatResponse(
            session_id=request.session_id,
            response=response.content
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error al procesar el mensaje: {str(e)}"
        )

In [ ]:
# 7. Endpoint POST /reset
@app.post("/reset", response_model=ResetResponse)
def reset(request: ResetRequest):
    # Si la sesión existe, la eliminamos
    if request.session_id in memory_store:
        del memory_store[request.session_id]
        return ResetResponse(session_id=request.session_id, status="Historial eliminado")
    else:
        # Si no existía, igual consideramos que está "reseteada"
        return ResetResponse(session_id=request.session_id, status="La sesión no tenía historial o ya estaba vacía")

In [ ]:
# 8. Levantar el servidor con uvicorn
import threading
import time
import uvicorn

PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

# Iniciar el servidor en un hilo separado para no bloquear el notebook
server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Servidor iniciado en el puerto {PORT}")

In [ ]:
# 9. Abrir túnel con ngrok
from pyngrok import ngrok

ngrok_authtoken = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError("Agrega NGROK_AUTHTOKEN en los Secrets de Colab.")

ngrok.set_auth_token(ngrok_authtoken)
ngrok.kill()  # Cerrar túneles anteriores para evitar conflictos
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("🌍 URL pública temporal:")
print(public_url)
print("\n📚 Documentación interactiva (Swagger UI):")
print(public_url + "/docs")

In [ ]:
import requests

headers = {"ngrok-skip-browser-warning": "true"}
base_url = public_url

# 1. Enviar mensaje a sesión 1
payload = {
    "session_id": "test1",
    "message": "Mi animal favorito es el perro."
}
resp = requests.post(f"{base_url}/chat", json=payload, headers=headers)
print("Respuesta 1:", resp.json())

# 2. Preguntar en la misma sesión
payload = {
    "session_id": "test1",
    "message": "¿Cuál es mi animal favorito?"
}
resp = requests.post(f"{base_url}/chat", json=payload, headers=headers)
print("Respuesta 2:", resp.json())
# Debería decir "perro"

# 3. Preguntar en otra sesión
payload = {
    "session_id": "test2",
    "message": "¿Cuál es mi animal favorito?"
}
resp = requests.post(f"{base_url}/chat", json=payload, headers=headers)
print("Respuesta 3 (otra sesión):", resp.json())
# No debería saberlo

# 4. Resetear sesión 1
payload = {"session_id": "test1"}
resp = requests.post(f"{base_url}/reset", json=payload, headers=headers)
print("Reset:", resp.json())

# 5. Preguntar nuevamente en sesión 1
payload = {
    "session_id": "test1",
    "message": "¿Cuál es mi animal favorito?"
}
resp = requests.post(f"{base_url}/chat", json=payload, headers=headers)
print("Respuesta después de reset:", resp.json())
# Ya no debería recordarlo